# dermaseg — Train SegFormer on Colab

Fine-tune **SegFormer (`nvidia/mit-b1`)** on **ISIC 2018 Task 1** (skin-lesion segmentation), evaluate on the held-out test split, and export an **ONNX** model for CPU serving.

> ⚠️ **Research / educational use only — not a medical device.** Trained on the public ISIC 2018 dataset; predictions can be wrong.

**Before you start**
- Use a **GPU runtime** (browser: *Runtime → Change runtime type → GPU*; VS Code: connect to a GPU Colab server).
- Push your latest code to GitHub first — the cells below clone from `origin`.
- The runtime filesystem is **ephemeral**; download `models/segformer.onnx` at the end.

## 0 · Check the GPU

In [1]:
!nvidia-smi

Tue May 26 07:17:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1 · Clone the repo & install training deps

In [2]:
!git clone https://github.com/onatakca/dermaseg.git
%cd dermaseg
!pip install -q -r requirements-train.txt

Cloning into 'dermaseg'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 54 (delta 5), reused 50 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 23.79 KiB | 23.79 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/dermaseg
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 138.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 115.7 MB/s eta 0:00:0000:0100:01


## 2 · Download ISIC 2018 (~11 GB)

Streams to the VM's local disk (a few minutes). Idempotent — safe to re-run.

In [3]:
!python scripts/download_data.py

[download] https://isic-challenge-data.s3.amazonaws.com/2018/ISIC2018_Task1-2_Training_Input.zip
  100.0%  10.4GB / 10.4GBGB
[extract] ISIC2018_Task1-2_Training_Input.zip -> images/
[done] 2594 files in /content/dermaseg/data/isic2018/images
[download] https://isic-challenge-data.s3.amazonaws.com/2018/ISIC2018_Task1_Training_GroundTruth.zip
  100.0%  26.1MB / 26.1MB
[extract] ISIC2018_Task1_Training_GroundTruth.zip -> masks/
[done] 2594 files in /content/dermaseg/data/isic2018/masks

Ready: 2594 images, 2594 masks under /content/dermaseg/data/isic2018


## 3 · Weights & Biases login (optional)

Paste your API key when prompted, or skip this cell and add `--no-wandb` to the training commands below.

In [4]:
import wandb
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: onatakca (onatakca1) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 4 · Smoke test (recommended)

A quick 2-epoch run to confirm the whole pipeline is green before the full job.

In [5]:
!python scripts/train.py --epochs 2 --batch-size 8 --no-wandb

Device: cuda
Split sizes: {'train': 1816, 'val': 389, 'test': 389}
config.json: 70.0kB [00:00, 79.5MB/s]
pytorch_model.bin: 100% 54.7M/54.7M [00:02<00:00, 27.0MB/s]
Loading weights: 100% 192/192 [00:00<00:00, 1210.48it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]            
SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b1
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.bias                               | UNEXPECTED | 
classifier.weight                             | UNEXPECTED | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head

## 5 · Full training (~50 epochs)

Saves the best checkpoint (by validation DICE) to `models/checkpoints/`. Logs to W&B unless you pass `--no-wandb`.

In [6]:
!python scripts/train.py --epochs 50 --batch-size 8

Device: cuda
Split sizes: {'train': 1816, 'val': 389, 'test': 389}
Loading weights: 100% 192/192 [00:00<00:00, 1134.39it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]            
SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b1
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.bias                               | UNEXPECTED | 
classifier.weight                             | UNEXPECTED | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.batch_norm.num_batches_tracked    |

## 6 · Evaluate on the test split

Writes `metrics.json` and fills the metrics table in `README.md`.

In [10]:
!python scripts/evaluate.py --checkpoint models/checkpoints --readme

Device: cuda | checkpoint: models/checkpoints
Loading weights: 100% 208/208 [00:00<00:00, 1101.12it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]            output.dense.weight]

test split (389 images):
| Metric | Score |
| --- | --- |
| DICE (lesion) | 0.9004 |
| Mean IoU | 0.8869 |
| Pixel accuracy | 0.9654 |

Wrote models/checkpoints/metrics.json
Updated metrics table in /content/dermaseg/README.md


## 7 · Export to ONNX (+ torch parity check)

In [12]:
!pip install -q onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 20.0 MB/s eta 0:00:00


In [13]:
!python scripts/export_onnx.py --checkpoint models/checkpoints --output models/segformer.onnx

Loading weights: 100% 208/208 [00:00<00:00, 724.86it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]             
Exporting models/checkpoints -> models/segformer.onnx (opset 17)
/content/dermaseg/scripts/export_onnx.py:59: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0526 09:33:21.501000 43991 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0526 09:33:

## 8 · Grab the artifacts

Download `models/segformer.onnx` (~20 MB) and commit it locally via git-lfs. In **VS Code**: right-click the file in the Explorer → *Download*. In the **browser**: run the cell below.

In [15]:
import os

print(f"ONNX size: {os.path.getsize('models/segformer.onnx') / 1e6:.1f} MB")
print(open('models/checkpoints/metrics.json').read())

# Browser Colab only — in VS Code use the Explorer → Download instead:
try:
    from google.colab import files

    files.download('models/segformer.onnx')
except Exception as e:
    print('Skipped files.download (not in browser Colab):', e)

ONNX size: 1.3 MB
{
  "split": "test",
  "n": 389,
  "dice": 0.9004318118095398,
  "miou": 0.8869457840919495,
  "pixel_acc": 0.9653685092926025
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Optional · Persist to Google Drive

Only needed to survive a disconnect *mid-training*. Mount Drive, then add `--output-dir /content/drive/MyDrive/dermaseg/ckpt` to the train command (and `--checkpoint /content/drive/MyDrive/dermaseg/ckpt` to eval/export).

- **VS Code:** Command Palette → `Colab: Mount Google Drive to Server` (known auth-prompt glitches — if it hangs, use the download approach above).
- **Browser:** `from google.colab import drive; drive.mount('/content/drive')`